In [7]:
import pandas as pd
import re
from google.colab import drive
import os

# --- Configuration ---
# 1. Define the input and output file names
INPUT_FILE_NAME = '25-10-14_Ref004_Edits.csv'
OUTPUT_FILE_NAME = '25-10-14_Ref004_Author_Extra_Info.csv'

# Define the regex to capture content within full-width Chinese brackets (including the brackets)
# r'（.*?）' means:
# （ and ）: Match the literal full-width brackets.
# .*: Match any character (.) zero or more times (*).
# ?: Make the match non-greedy (stop at the *first* closing bracket).
CHINESE_BRACKET_REGEX = r'[（(].*?[)）]'

# --- Step 1: Mount Google Drive ---
# You will be prompted to authorize Google Drive access.
print("Mounting Google Drive...")
drive.mount('/content/drive')

# --- Step 2: Define the file paths ---
# IMPORTANT: Adjust the 'MyDrive/' path if your file is in a subdirectory.
# This assumes the file is in the root of your 'My Drive'.
input_path = os.path.join('/content/drive/MyDrive', INPUT_FILE_NAME)
output_path = os.path.join('/content/drive/MyDrive', OUTPUT_FILE_NAME)

# --- Step 3: Load the CSV file ---
try:
    print(f"Loading file from: {input_path}")
    df = pd.read_csv(input_path)
    print("File loaded successfully. Processing data...")
except FileNotFoundError:
    print(f"\nERROR: File not found at {input_path}")
    print("Please check the file name and path (e.g., if it's in a subfolder like 'My Drive/Data/').")
    # Exit or handle error
    exit()

# --- Step 4: Extract the extra information ---

# Use .str.findall() to find ALL occurrences of the regex pattern in the 'Author' column.
# This results in a Series where each element is a LIST of matched strings (e.g., ['（info1）', '（info2）']).
extra_info_list_series = df['Author'].astype(str).str.findall(CHINESE_BRACKET_REGEX)

# Join the list of extracted strings into a single string, separated by a space,
# and create the new "Author_extra_info" column.
# Changed the separator from ' ' to '' to remove the space between extracted information.
df['Author_extra_info'] = extra_info_list_series.apply(lambda x: ''.join(x) if x else '')

# --- Step 5: Create "Author_no_extra_info" (Removal) ---
print("Creating 'Author_no_extra_info' column (Removing data in brackets)...")

# Use the compiled pattern to replace all matched bracketed text with an empty string ('')
df['Author_no_extra_info'] = df['Author'].str.replace(CHINESE_BRACKET_REGEX, '', regex=True)

# Clean up any leftover leading/trailing whitespace after removal
df['Author_no_extra_info'] = df['Author_no_extra_info'].str.strip()


# Verification:
# As requested, the original 'Author' column remains the same (unmodified).
print(f"\nOriginal 'Author' column remains unchanged.")
print(f"New column 'Author_extra_info' created.")

# --- Step 5: Save the new DataFrame to a CSV file ---

print(f"\nSaving new file to: {output_path}")
# The new DataFrame is saved to the specified location in Google Drive.
df.to_csv(output_path, index=False)
print("Processing complete. The new file has been successfully saved.")

# Optional: Display the first few rows of the result
print("\nFirst 5 rows of the resulting DataFrame:")
print(df[['Author', 'Author_extra_info']].head())

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading file from: /content/drive/MyDrive/25-10-14_Ref004_Edits.csv
File loaded successfully. Processing data...


/tmp/ipython-input-3795513416.py:32: DtypeWarning: Columns (11,12,13,14,17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_path)


Creating 'Author_no_extra_info' column (Removing data in brackets)...

Original 'Author' column remains unchanged.
New column 'Author_extra_info' created.

Saving new file to: /content/drive/MyDrive/25-10-14_Ref004_Author_Extra_Info.csv
Processing complete. The new file has been successfully saved.

First 5 rows of the resulting DataFrame:
    Author Author_extra_info
0      赵伯衡                  
1  董均伦、张文元                  
2      NaN                  
3      NaN                  
4    野夫、江丰                  


In [9]:
import pandas as pd
from google.colab import drive
import os

# --- STEP 1: Mount Google Drive ---
# This connects Colab to your Google Drive to access the file.
print("Mounting Google Drive...")
drive.mount('/content/drive')

# --- STEP 2: Define File Paths ---
# IMPORTANT: You MUST replace 'Your_Folder/' with the actual path to your file.
# Example: '/content/drive/MyDrive/Colab Data/'
DRIVE_PATH = '/content/drive/MyDrive/'

INPUT_FILENAME = "25-10-14_Ref004_Edits.csv"
OUTPUT_FILENAME = "25-10-14_Ref004_Author_Core_Split.csv"

INPUT_FILEPATH = os.path.join(DRIVE_PATH, INPUT_FILENAME)
OUTPUT_FILEPATH = os.path.join(DRIVE_PATH, OUTPUT_FILENAME)

try:
    # --- STEP 3: Load the CSV file ---
    df = pd.read_csv(INPUT_FILEPATH)
    print(f"Successfully loaded '{INPUT_FILENAME}'. Original shape: {df.shape}")

    # --- STEP 4: Split the "Author_Core" column ---
    # Regex breakdown:
    # [/\s、] matches a forward slash (/), any whitespace character (\s), OR the Chinese enumeration comma (、).
    # expand=True creates a new DataFrame with the split results in separate columns.
    # Removed n=4 to split into as many columns as needed.

    # NOTE: The pandas split operation inherently removes the delimiters.
    split_df = df['Author_Core'].str.split(r'[/\s、]', expand=True)

    # --- STEP 5: Rename the new columns ---
    # Create a list of the desired new column names
    # Since the number of columns is now dynamic, we'll generate names based on the number of columns created.
    new_cols = [f"Author_{i+1}" for i in range(split_df.shape[1])]

    # Rename the columns in the split DataFrame
    split_df.columns = new_cols

    # --- STEP 6: Merge the new columns back to the original DataFrame ---
    # The 'Author_Core' column remains unchanged in the original df.
    df = pd.concat([df, split_df], axis=1)

    print("\nSplit complete. New columns added to the DataFrame.")
    print(f"Updated DataFrame shape: {df.shape}")
    # Display the relevant columns dynamically
    print(f"Check the first few entries of the split data:\n{df[['Author_Core'] + new_cols].head()}")


    # --- STEP 7: Save the new CSV file ---
    df.to_csv(OUTPUT_FILEPATH, index=False)
    print(f"\n✅ Success! New CSV file saved as '{OUTPUT_FILENAME}' to: {DRIVE_PATH}")

except FileNotFoundError:
    print(f"\n❌ ERROR: File not found at '{INPUT_FILEPATH}'. Please check the DRIVE_PATH and file name.")
except KeyError:
    print("\n❌ ERROR: Column 'Author_Core' not found. Please verify the column name in your CSV.")
except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipython-input-2708275055.py:23: DtypeWarning: Columns (13,14,15,16,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(INPUT_FILEPATH)


Successfully loaded '25-10-14_Ref004_Edits.csv'. Original shape: (35424, 21)

Split complete. New columns added to the DataFrame.
Updated DataFrame shape: (35424, 27)
Check the first few entries of the split data:
  Author_Core Author_1 Author_2 Author_3 Author_4 Author_5 Author_6
0         赵伯衡      赵伯衡     None     None     None     None     None
1     董均伦、张文元      董均伦      张文元     None     None     None     None
2         NaN      NaN      NaN      NaN      NaN      NaN      NaN
3         NaN      NaN      NaN      NaN      NaN      NaN      NaN
4       野夫、江丰       野夫       江丰     None     None     None     None

✅ Success! New CSV file saved as '25-10-14_Ref004_Author_Core_Split.csv' to: /content/drive/MyDrive/
